In [ ]:
# 相场数据流水线 — 改RUN_START/RUN_END一口气跑多组 (Kaggle)
import os
import numpy as np
import cupy as cp
from tqdm import tqdm
from scipy import ndimage

# =======================================================================
# 空间网格敏感性：1024² @ 1 nm，固定 1024 nm 物理域和参考长度
# =======================================================================
nx, ny = 1024, 1024
dx_real = 1e-9
dt = 5e-4
total_steps = 504000
save_interval = 2000

c_gamma  = 0.10
c_gprime = 0.22
c_matrix = 0.13

gamma_int = 0.091
gamma_gb  = 1.02
delta     = 0.0072
C11 = 174.9e9
C12 = 117.6e9
C44 = 71.6e9

f0 = 1.27e9
length_scale = 2e-9  # 固定参考长度；细化网格不改变无量纲化
h = dx_real / length_scale
dx = length_scale
lam = 5.0
dc = c_gprime - c_gamma

koA = (lam * dc / 4) ** 2
gid = gamma_int / (f0 * dx)
gbd = gamma_gb  / (f0 * dx)
ktA = (gid * 6 / dc**3) ** 2
A_well  = np.sqrt(ktA / koA)
kappa_c = koA * A_well

D_bulk = 10.0
D_gb   = 30.0
S_stab = D_bulk * A_well
elastic_scale = 150.0
GB_penalty = 50.0

alpha_pinning = 25.0
beta_zener = 1500.0

W_gb = 1000.0
kappa_eta = 160.0
L_mobility = 0.4
num_grains = 6

assert abs(nx * dx_real - 1024e-9) < 1e-18
assert abs(total_steps * dt - 252.0) < 1e-12
assert abs(save_interval * dt - 1.0) < 1e-12

# =======================================================================
# ★ 改这两个数字一口气跑多组
# =======================================================================
RUN_START = 31
RUN_END   = 31

for run_id in range(RUN_START, RUN_END + 1):
    output_dir = "/kaggle/working/Spatial_1024_dx1nm_Run_{:02d}".format(run_id)
    os.makedirs(output_dir, exist_ok=True)
    print("=" * 60)
    print("Run {:02d} -> {}".format(run_id, output_dir))

    # ---- 同一物理初态：先复现原始 512² realization，再映射到 1024² ----
    # 原始随机场在 512² 网格上生成；细网格通过周期三次插值采样同一场。
    # 位移、晶粒中心、种子中心及半径随后按 grid_scale=2 映射。
    coarse_nx = coarse_ny = 512
    grid_scale = 2
    assert nx == coarse_nx * grid_scale and ny == coarse_ny * grid_scale

    grain_seeds_coarse = [(100,150),(300,100),(450,220),(380,420),(180,400),(80,280)]
    yg0, xg0 = cp.indices((coarse_ny, coarse_nx))
    kx0 = cp.fft.fftfreq(coarse_nx) * 2 * np.pi
    ky0 = cp.fft.fftfreq(coarse_ny) * 2 * np.pi
    K20 = kx0[:, None]**2 + ky0[None, :]**2

    cp.random.seed(777 + run_id)
    noise_x0 = cp.random.randn(coarse_ny, coarse_nx) + 1j * cp.random.randn(coarse_ny, coarse_nx)
    noise_x0 *= cp.exp(-K20 * 60**2 / 2)
    fx0 = cp.real(cp.fft.ifft2(noise_x0))
    fx0 *= 25.0 / cp.std(fx0)

    cp.random.seed(888 + run_id)
    noise_y0 = cp.random.randn(coarse_ny, coarse_nx) + 1j * cp.random.randn(coarse_ny, coarse_nx)
    noise_y0 *= cp.exp(-K20 * 60**2 / 2)
    fy0 = cp.real(cp.fft.ifft2(noise_y0))
    fy0 *= 25.0 / cp.std(fy0)

    xgw0 = xg0 + fx0
    ygw0 = yg0 + fy0
    ds0 = []
    for sx, sy in grain_seeds_coarse:
        ddx = xgw0 - sx
        ddx -= cp.round(ddx / coarse_nx) * coarse_nx
        ddy = ygw0 - sy
        ddy -= cp.round(ddy / coarse_ny) * coarse_ny
        ds0.append(ddx**2 + ddy**2)
    gid0 = cp.argmin(cp.array(ds0), axis=0)

    # 在 CPU 上对仅两个 512² 位移场做周期插值，避免改变随机 realization。
    sample_y, sample_x = np.meshgrid(
        np.arange(ny, dtype=np.float64) / grid_scale,
        np.arange(nx, dtype=np.float64) / grid_scale,
        indexing="ij")
    coords = np.stack([sample_y, sample_x], axis=0)
    fx_fine = ndimage.map_coordinates(cp.asnumpy(fx0), coords, order=3, mode="wrap") * grid_scale
    fy_fine = ndimage.map_coordinates(cp.asnumpy(fy0), coords, order=3, mode="wrap") * grid_scale
    del sample_y, sample_x, coords, noise_x0, noise_y0

    yg, xg = cp.indices((ny, nx))
    xgw = xg + cp.asarray(fx_fine)
    ygw = yg + cp.asarray(fy_fine)
    del fx_fine, fy_fine

    grain_seeds = [(sx * grid_scale, sy * grid_scale) for sx, sy in grain_seeds_coarse]
    ds = []
    for sx, sy in grain_seeds:
        ddx = xgw - sx
        ddx -= cp.round(ddx / nx) * nx
        ddy = ygw - sy
        ddy -= cp.round(ddy / ny) * ny
        ds.append(ddx**2 + ddy**2)
    gid = cp.argmin(cp.array(ds), axis=0)
    del ds, xgw, ygw

    etas = cp.zeros((num_grains, ny, nx))
    for i in range(num_grains):
        etas[i] = cp.where(gid == i, 1.0, 0.0)

    # h=0.5，sigma=1 个参考长度单位等价于细网格 2 pixels。
    kx_init = cp.fft.fftfreq(nx, d=h) * 2 * np.pi
    ky_init = cp.fft.fftfreq(ny, d=h) * 2 * np.pi
    K2_init = kx_init[:, None]**2 + ky_init[None, :]**2
    for i in range(num_grains):
        etas[i] = cp.real(cp.fft.ifft2(
            cp.fft.fft2(etas[i]) * cp.exp(-K2_init * 1.0**2 / 2)))
    del K2_init

    # ---- 撒种：在粗网格复现原始随机选择，再将物理中心映射到细网格 ----
    c = cp.full((ny, nx), c_matrix)
    gbx0 = gid0 != cp.roll(gid0, 1, axis=1)
    gby0 = gid0 != cp.roll(gid0, 1, axis=0)
    gbi0 = cp.where(gbx0 | gby0)
    cp.random.seed(99 + run_id * 10)
    chosen0 = cp.random.choice(cp.arange(len(gbi0[0])), size=60, replace=False)
    offsets = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(1,-1),(-1,1),(1,1)]
    for k in chosen0:
        iy0 = int(gbi0[0][k])
        ix0 = int(gbi0[1][k])
        owner = int(gid0[iy0, ix0])
        best_count = best_dx = best_dy = 0
        for odx, ody in offsets:
            count = sum(
                1 for s in range(1, 8)
                if 0 <= iy0 + s*ody < coarse_ny
                and 0 <= ix0 + s*odx < coarse_nx
                and int(gid0[iy0 + s*ody, ix0 + s*odx]) == owner)
            if count > best_count:
                best_count, best_dx, best_dy = count, odx, ody
        cx = (ix0 + best_dx * 6) * grid_scale
        cy = (iy0 + best_dy * 6) * grid_scale
        # 原始近 GB 半径 4 coarse pixels = 8 nm = 8 fine pixels。
        c = cp.where(
            (gid == owner) & ((xg - cx)**2 + (yg - cy)**2 < (4 * grid_scale)**2),
            c_gprime, c)

    cp.random.seed(123 + run_id * 13)
    for g in range(num_grains):
        gy0, gx0 = cp.where(gid0 == g)
        for _ in range(60):
            idx0 = int(cp.random.randint(0, len(gx0), 1)[0])
            sx = int(gx0[idx0]) * grid_scale
            sy = int(gy0[idx0]) * grid_scale
            # 原始晶内半径 2 coarse pixels = 4 nm = 4 fine pixels。
            c = cp.where(
                (gid == g) & ((xg - sx)**2 + (yg - sy)**2 < (2 * grid_scale)**2),
                c_gprime, c)
    del gid0, gid, gbi0, gbx0, gby0
    del fx0, fy0, ds0, xgw0, ygw0, xg0, yg0, xg, yg, K20, kx0, ky0
    cp.get_default_memory_pool().free_all_blocks()

    # ---- 弹性核 B(k) ----
    kx = cp.fft.fftfreq(nx, d=h) * 2 * np.pi
    ky = cp.fft.fftfreq(ny, d=h) * 2 * np.pi
    KX, KY = cp.meshgrid(kx, ky)
    K2 = KX**2 + KY**2
    K2s = cp.where(K2 == 0, 1.0, K2)
    K4r = (KX**4 + KY**4) / K2s**2

    AZ = 2 * C44 / (C11 - C12)
    an = (AZ - 1) / (AZ + 1)
    B0 = elastic_scale * (C11 / f0) * delta**2

    ga = [0.0, np.pi/6, np.pi/3, np.pi/2, 2*np.pi/3, 5*np.pi/6]
    BK = cp.zeros((num_grains, ny, nx))
    for idx, th in enumerate(ga):
        KXr = KX * cp.cos(th) + KY * cp.sin(th)
        KYr = -KX * cp.sin(th) + KY * cp.cos(th)
        K4rr = (KXr**4 + KYr**4) / K2s**2
        Bk = -B0 * (1.0 + an * K4rr)
        Bk[0, 0] = 0.0
        BK[idx] = Bk

    # ---- 预计算 ----
    ak4 = K2**2
    lhs_e = 1.0 + dt * L_mobility * kappa_eta * K2
    lhs_c = 1.0 + dt * S_stab * ak4

    def get_phi(etas_in):
        return cp.clip((1.0 - cp.sum(etas_in**2, axis=0)) * 2.0, 0.0, 1.0)

    phi = get_phi(etas)
    Deff = D_bulk * (1 - phi) + D_gb * phi

    def laplacian(f):
        return cp.real(cp.fft.ifft2(-K2 * cp.fft.fft2(f)))

    # 保存未推进的共同初态，供离散设置之间核对；不参与时间推进。
    initial7 = np.concatenate([
        cp.asnumpy(c).astype(np.float32)[None, :, :],
        cp.asnumpy(etas).astype(np.float32)], axis=0)
    np.save(os.path.join(output_dir, 'initial_state_T000_float32.npy'), initial7)
    del initial7

    t_hist = []
    r_hist = []
    v_hist = []
    l_hist = []

    # ---- 主循环 ----
    for step in tqdm(range(1, total_steps + 1), desc="Run{:02d}".format(run_id)):

        # Cahn-Hilliard
        fc = A_well * (c - c_gamma) * (c - c_gprime) * (2*c - c_gamma - c_gprime)
        cf = cp.fft.fft2(c)
        dE = cp.zeros((ny, nx))
        for i in range(num_grains):
            dE += cp.real(cp.fft.ifft2(BK[i] * cf)) * etas[i]
        mu = fc - kappa_c * laplacian(c) + dE + GB_penalty * phi * (c - c_gamma)
        Dx = 0.5 * (Deff + cp.roll(Deff, -1, 1))
        fx = Dx * (cp.roll(mu, -1, 1) - mu) / h
        Dy = 0.5 * (Deff + cp.roll(Deff, -1, 0))
        fy = Dy * (cp.roll(mu, -1, 0) - mu) / h
        df = ((fx - cp.roll(fx, 1, 1)) + (fy - cp.roll(fy, 1, 0))) / h
        cf = cp.fft.fft2(c)
        rc = cf + dt * cp.fft.fft2(df) + dt * S_stab * ak4 * cf
        c = cp.real(cp.fft.ifft2(rc / lhs_c))
        c += (c_matrix - cp.mean(c))
        c = cp.clip(c, 0.08, 0.24)

        # Allen-Cahn + Zener
        Wgv = W_gb * (1.0 + alpha_pinning * cp.clip(c - c_gamma, 0, 1))
        zf = beta_zener * cp.clip(c - c_gamma, 0, 1) ** 2
        ss = cp.sum(etas**2, axis=0)
        for i in range(num_grains):
            sj = ss - etas[i]**2
            sw = etas[i] * (etas[i]**2 - 1) + 2 * etas[i] * sj
            sz = zf * etas[i] * (1 - etas[i]) * (etas[i] - 0.5)
            ef = cp.fft.fft2(etas[i])
            sf = cp.fft.fft2(-L_mobility * (Wgv * sw + sz))
            etas[i] = cp.real(cp.fft.ifft2((ef + dt * sf) / lhs_e))
            etas[i] = cp.clip(etas[i], 0.0, 1.0)
        phi = get_phi(etas)
        Deff = D_bulk * (1 - phi) + D_gb * phi

        # 保存 + 诊断 (7ch: c + eta1 + ... + eta6)
        if step % save_interval == 0:
            if not bool(cp.all(cp.isfinite(c))) or not bool(cp.all(cp.isfinite(etas))):
                raise FloatingPointError(f'Non-finite field detected at step {step}')
            cn = cp.asnumpy(c).astype(np.float16)
            en = cp.asnumpy(etas).astype(np.float16)
            frame7 = np.concatenate([cn[None, :, :], en], axis=0)
            assert frame7.shape == (7, ny, nx)
            np.save(
                os.path.join(output_dir, "frame_{:06d}.npy".format(step)),
                frame7)  # (7, H, W): c + eta1 + ... + eta6

            vol = float(cp.mean(c > 0.18))
            mask = cn > 0.18
            labeled, nl = ndimage.label(mask)
            if nl > 0:
                areas = ndimage.sum(mask, labeled, range(1, nl + 1))
                rm = np.sqrt(np.mean(areas) / np.pi) * dx_real * 1e9
            else:
                rm = 0.0

            pg = get_phi(etas)
            gx = (cp.roll(pg, -1, 1) - cp.roll(pg, 1, 1)) * 0.5
            gy = (cp.roll(pg, -1, 0) - cp.roll(pg, 1, 0)) * 0.5
            gl = float(0.5 * cp.sum(cp.sqrt(gx**2 + gy**2))) * dx_real * 1e9

            t_hist.append(float(step * dt))
            r_hist.append(float(rm))
            v_hist.append(float(vol))
            l_hist.append(float(gl))
            tqdm.write("T*={:.0f} r={:.1f}nm vol={:.3f} Lgb={:.0f}nm".format(
                       step * dt, rm, vol, gl))

    dp = os.path.join(output_dir, "diagnostic_data.txt")
    np.savetxt(dp,
               np.column_stack([np.array(t_hist),
                                np.array(r_hist),
                                np.array(v_hist),
                                np.array(l_hist)]),
               header="T* r_mean(nm) vol_frac L_gb(nm)",
               fmt="%.6f %.4f %.4f %.1f")
    print("Run{:02d} 模拟计算完成! 正在写入最后的数据: {}".format(run_id, output_dir))

    # ---- 防丢补丁：跑完立刻打包 ----
    from IPython.display import display, FileLink

    print("📦 正在后台将数据打包为 zip，请稍等几秒钟...")
    zip_filename = "Spatial_1024_dx1nm_Run_{:02d}.zip".format(run_id)
    zip_filepath = os.path.join("/kaggle/working", zip_filename)
    import shutil
    archive_base = os.path.splitext(zip_filepath)[0]
    shutil.make_archive(archive_base, "zip", root_dir=os.path.dirname(output_dir), base_dir=os.path.basename(output_dir))

    if os.path.exists(zip_filepath):
        print("✨ 打包成功！点击下方蓝色链接下载：")
        os.chdir('/kaggle/working')
        display(FileLink(zip_filename))
    else:
        print("❌ 警告：打包似乎失败了，请检查输出日志。")

print("=" * 60)
print("全部 {} 组数据集生成完毕!".format(RUN_END - RUN_START + 1))
print("=" * 60)